No WS4 command here uses shell magic: every one goes through this helper, which is also
how you would drive the command line from any script.

This one is the **library** path for the loop, with one compile through the library.


In [1]:
import subprocess


def run(command: str) -> None:
    """Run a WS4 command, printing what it printed."""
    result = subprocess.run(command, shell=True, text=True, capture_output=True)
    print((result.stdout or result.stderr).strip())
    result.check_returncode()

# Sweep a parameter

The command line is the fastest way to run one study; the Python API is the way to run many.
This notebook sweeps one number — the floor on harvested volume — and collects the results into
a frame, which is the shape of most sensitivity work.

## The loop

Every run writes the same files, so a sweep is a loop that reads `summary.json` and the target
report back.

In [2]:
import json
from pathlib import Path

import pandas as pd
from ws4.build.compiler import build_canonical_dataset
from ws4.opt.engine import OptimizeConfig, Optimizer

work = Path("tmp/examples/sweep")
work.mkdir(parents=True, exist_ok=True)
dataset = build_canonical_dataset("examples/estate/recipe.xml", "examples/estate/fragments.csv", work / "compiled")
print(dataset)

tmp/examples/sweep/compiled


In [3]:
rows = []
for floor in (10_000, 20_000, 30_000):
    targets = work / f"targets-{floor}.yaml"
    targets.write_text(
        "targets:\n"
        "- account: product.Harvested.CC\n"
        "  period: 1\n"
        f"  minimum: {floor}\n"
        "  period_width: 10\n"
        "- account: patch.opening.50_150.size\n"
        "  period: 1\n"
        "  minimum: 30\n"
        "  min_weight: 2000\n"
        "  shape: non_linear\n"
        "  period_width: 10\n"
    )
    out = work / f"run-{floor}"
    optimizer = Optimizer.from_dataset(
        dataset,
        OptimizeConfig(years=100, iterations=2000, seed=1, targets_path=str(targets),
                       topology_path="examples/estate/topology.csv",
                       patch_accounts_path="examples/estate/patches.yaml",
                       merge_weight=0.5, split_weight=0.2, cluster_weight=0.3, threads=1),
    )
    summary = optimizer.run(out)
    report = pd.read_csv(out / "report_targets.csv")
    rows.append({
        "floor": floor,
        "objective": round(summary["incumbent_objective"], 1),
        "treatments": summary["blocks_scheduled"],
        "volume_share": float(report.loc[report.account == "product.Harvested.CC", "value"].iloc[0]),
        "patch_met": report.loc[report.account.str.startswith("patch"), "status"].iloc[0],
    })

print(pd.DataFrame(rows).to_string(index=False))

 floor  objective  treatments  volume_share patch_met
 10000        0.0          12  12156.000000     below
 20000       -0.0          18  20848.200000     below
 30000       -0.0          29  30218.142857     below


The frame is the point: one row per configuration, with the objective and the target statuses
beside the parameter. A spread in the objective across seeds, or a target that flips between
`met` and `below` between neighbouring settings, is what tells you whether a result is worth
reporting.

## Where to look next

* `docs/howto/automate-a-study.md` collects the patterns: in-memory targets, hand-driven solvers,
  and the island model.
* `docs/reference/optimizer.rst` specifies the objective and the search stack.